# Projeto parcial de classificação do atraso na partida

**Disciplina:** Introdução ao Machine Learning  
**Integrantes:** Francisco Alberto Fonseca Neto, Lucimar Oliveira do Nascimento e Osvaldo Souza Menezes Júnior  
**Representante:** Lucimar Oliveira do Nascimento

## 1. Problema, escopo e variável-alvo

**Pergunta de pesquisa:** em que medida informações disponíveis antes da partida permitem classificar a faixa de atraso na partida dos voos realizados registrados na base VRA/ANAC de 2024 e 2025?

Esta primeira atividade caracteriza a base, audita sua qualidade, define a população elegível e descreve o alvo. Ela não treina modelos nem mede desempenho preditivo. Cancelamentos constituem outro desfecho e ficam fora da população que recebe classe de atraso.

O alvo principal é `codigo_faixa_atraso`, acompanhado do rótulo `faixa_atraso_partida`. Ambos derivam de `atraso_partida_minutos = partida_real − partida_prevista`. O indicador secundário `atraso_bi` vale 1 para atrasos de pelo menos 15 minutos.


## 2. Ambiente de execução

As bibliotecas abaixo cobrem leitura, transformação, tabelas e gráficos. A configuração visual não altera os dados. Todos os resultados são produzidos pela execução; o notebook entregue permanece sem outputs persistidos.


In [ ]:
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda valor: f'{valor:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')


## 3. Localização determinística da base

O pacote final coloca o notebook e o ZIP lado a lado. Durante o desenvolvimento, a mesma base fica em `data/raw/`. Os locais são avaliados por prioridade: uma base ao lado do notebook sempre prevalece sobre a cópia interna do repositório. Mais de um candidato no mesmo nível é tratado como ambiguidade e interrompe a execução.

Quando a entrada é ZIP, o arquivo deve conter exatamente o CSV consolidado esperado. Nenhum conteúdo é extraído permanentemente.


In [ ]:
grupos_candidatos = [
    [
        Path('base_lucimar_oliveira_do_nascimento.csv'),
        Path('base_lucimar_oliveira_do_nascimento.zip'),
        Path('base_lucimar_nascimento_v2.csv'),
        Path('base_lucimar_nascimento_v2.zip'),
    ],
    [
        Path('data/raw/base_lucimar_nascimento_v2.zip'),
        Path('../../data/raw/base_lucimar_nascimento_v2.zip'),
    ],
]

caminho_base = None
for grupo in grupos_candidatos:
    encontrados = [caminho for caminho in grupo if caminho.is_file()]
    if len(encontrados) > 1:
        raise FileExistsError(f'Mais de uma base no mesmo nível: {encontrados}')
    if encontrados:
        caminho_base = encontrados[0]
        break
if caminho_base is None:
    raise FileNotFoundError('Nenhuma base oficial foi encontrada nos locais aceitos.')

membro_csv = None
if caminho_base.suffix.lower() == '.zip':
    with ZipFile(caminho_base) as arquivo_zip:
        membros_csv = [nome for nome in arquivo_zip.namelist() if nome.lower().endswith('.csv')]
    if membros_csv != ['base_lucimar_nascimento_v2.csv']:
        raise ValueError(f'Conteúdo inesperado no ZIP: {membros_csv}')
    membro_csv = membros_csv[0]

print('Arquivo escolhido:', caminho_base)
print('CSV interno:', membro_csv if membro_csv else caminho_base.name)


## 4. Leitura e validação do esquema consolidado

A base consolidada possui 25 campos. A validação exige correspondência exata, pois continuar com colunas ausentes ou extras poderia produzir análises incompletas. Identificadores são lidos como texto para preservar códigos e zeros à esquerda.


In [ ]:
COLUNAS_CSV = [
    'companhia_icao', 'empresa', 'numero_voo', 'codigo_di', 'codigo_tipo_linha',
    'modelo_equipamento', 'numero_assentos', 'origem_icao', 'origem_descricao',
    'partida_prevista', 'partida_real', 'destino_icao', 'destino_descricao',
    'chegada_prevista', 'chegada_real', 'situacao_voo', 'situacao_partida',
    'situacao_chegada', 'codeshare', 'cancelado', 'realizado', 'atraso_partida_min',
    'atraso_chegada_min', 'arquivo_origem', 'linha_origem',
]
IDENTIFICADORES = [
    'companhia_icao', 'numero_voo', 'codigo_di', 'codigo_tipo_linha',
    'modelo_equipamento', 'origem_icao', 'destino_icao', 'codeshare',
]

df_consolidada = pd.read_csv(
    caminho_base,
    sep=',',
    encoding='utf-8-sig',
    dtype={coluna: 'string' for coluna in IDENTIFICADORES},
    low_memory=False,
)

faltantes = set(COLUNAS_CSV) - set(df_consolidada.columns)
extras = set(df_consolidada.columns) - set(COLUNAS_CSV)
if faltantes or extras or list(df_consolidada.columns) != COLUNAS_CSV:
    raise ValueError(f'Esquema inesperado. Faltam: {faltantes}; extras: {extras}')
if df_consolidada.empty:
    raise ValueError('A base consolidada não possui registros.')

print(f'Dimensões: {len(df_consolidada):,} linhas × {len(df_consolidada.columns)} colunas.')
display(pd.DataFrame({'Coluna recebida': df_consolidada.columns}, index=range(1, 26)))
display(df_consolidada.head(3))


O resultado esperado é uma tabela com as 25 colunas na ordem oficial e uma amostra de registros. Qualquer divergência interrompe a execução antes de gráficos ou conclusões.


## 5. Perfil inicial da base consolidada

Antes de filtros ou conversões, examinamos tipos inferidos, ausências e possíveis repetições. A ausência é apresentada em quantidade e percentual; tipo `object` ou `string` neste estágio não significa que o campo já tenha sido validado semanticamente.


In [ ]:
perfil_inicial = pd.DataFrame({
    'Tipo inferido': df_consolidada.dtypes.astype(str),
    'Ausentes': df_consolidada.isna().sum(),
    'Ausentes (%)': df_consolidada.isna().mean().mul(100).round(2),
    'Categorias distintas': df_consolidada.nunique(dropna=True),
}).sort_values(['Ausentes (%)', 'Categorias distintas'], ascending=[False, False])

display(perfil_inicial)
print('Duplicidades exatas:', int(df_consolidada.duplicated().sum()))
print(
    'Repetições de arquivo_origem + linha_origem:',
    int(df_consolidada.duplicated(['arquivo_origem', 'linha_origem']).sum()),
)


### Situações operacionais e domínios declarados

Os campos de situação e os indicadores de cancelamento e realização são auditados antes de definir a população. Valores inesperados não são convertidos silenciosamente para verdadeiro ou falso.


In [ ]:
for coluna in ['situacao_voo', 'cancelado', 'realizado']:
    frequencias = (
        df_consolidada[coluna]
        .astype('string')
        .str.strip()
        .fillna('(ausente)')
        .value_counts(dropna=False)
        .rename('Registros')
        .to_frame()
    )
    print(f'Domínio observado — {coluna}:')
    display(frequencias)

print('Cardinalidade dos principais identificadores:')
display(
    df_consolidada[['companhia_icao', 'origem_icao', 'destino_icao', 'numero_voo']]
    .nunique(dropna=True)
    .rename('Categorias distintas')
    .to_frame()
)


Ausências, valores inesperados e repetições são sinais para auditoria, não justificativas automáticas para descarte. A proveniência `arquivo_origem + linha_origem` ajuda a rastrear registros, mas não é assumida como chave de negócio.


## 6. Padronização dos nomes analíticos

O mapeamento abaixo é específico para a base entregue. Ele cria nomes usados na análise sem alterar o objeto `df_consolidada`. Colunas obrigatórias ausentes ou nomes duplicados após a renomeação interrompem a execução.


In [ ]:
MAPEAMENTO = {
    'companhia_icao': 'sg_empresa_icao',
    'numero_voo': 'nr_voo',
    'origem_icao': 'sg_icao_origem',
    'destino_icao': 'sg_icao_destino',
    'partida_prevista': 'dt_partida_prevista',
    'partida_real': 'dt_partida_real',
    'chegada_prevista': 'dt_chegada_prevista',
    'chegada_real': 'dt_chegada_real',
    'situacao_voo': 'cd_situacao_voo',
}

if not set(MAPEAMENTO).issubset(df_consolidada.columns):
    raise ValueError('A base não contém todas as colunas exigidas pelo mapeamento.')
df_padronizado = df_consolidada.rename(columns=MAPEAMENTO)
if df_padronizado.columns.duplicated().any():
    duplicadas = df_padronizado.columns[df_padronizado.columns.duplicated()].tolist()
    raise ValueError(f'Nomes duplicados após o mapeamento: {duplicadas}')

print('Mapeamento validado:', len(MAPEAMENTO), 'colunas renomeadas.')
display(pd.Series(MAPEAMENTO, name='Nome analítico').rename_axis('Nome na base').to_frame())


## 7. Conversão e auditoria dos timestamps

As quatro colunas temporais usam o formato explícito `AAAA-MM-DD HH:MM:SS`. A auditoria distingue valores originalmente ausentes de valores preenchidos que não puderam ser convertidos.

Os metadados oficiais do VRA informam que horários previstos e realizados são publicados no horário de Brasília. Isso permite comparar partida prevista e real diretamente, mas não autoriza corrigir automaticamente datas inconsistentes.


In [ ]:
df_auditado = df_padronizado.copy()
COLUNAS_TEMPORAIS = [
    'dt_partida_prevista', 'dt_partida_real',
    'dt_chegada_prevista', 'dt_chegada_real',
]
auditoria_conversao = []

for coluna in COLUNAS_TEMPORAIS:
    original = df_auditado[coluna].astype('string').str.strip().replace('', pd.NA)
    convertido = pd.to_datetime(original, format='%Y-%m-%d %H:%M:%S', errors='coerce')
    auditoria_conversao.append({
        'Coluna': coluna,
        'Ausentes originalmente': int(original.isna().sum()),
        'Falhas de conversão': int((original.notna() & convertido.isna()).sum()),
        'Menor timestamp válido': convertido.min(),
        'Maior timestamp válido': convertido.max(),
    })
    df_auditado[coluna] = convertido

display(pd.DataFrame(auditoria_conversao))
print('Tipos após a conversão:')
display(df_auditado[COLUNAS_TEMPORAIS].dtypes.astype(str).rename('Tipo').to_frame())


## 8. Duplicidades por chave natural candidata

Além de duplicidades exatas e da chave de proveniência, investigamos uma chave de negócio candidata composta por companhia, número do voo, origem, destino e partida prevista. Ela diferencia datas distintas porque o timestamp previsto faz parte da chave, mas ainda pode reunir registros legítimos de uma mesma etapa operacional. Por isso, os resultados são apenas sinalizados.


In [ ]:
CHAVE_NATURAL = [
    'sg_empresa_icao', 'nr_voo', 'sg_icao_origem',
    'sg_icao_destino', 'dt_partida_prevista',
]
chave_completa = df_auditado[CHAVE_NATURAL].notna().all(axis=1)
duplicada_chave = chave_completa & df_auditado.duplicated(CHAVE_NATURAL, keep=False)

grupos_duplicados = (
    df_auditado.loc[duplicada_chave]
    .groupby(CHAVE_NATURAL, dropna=False)
    .size()
    .rename('Registros no grupo')
    .sort_values(ascending=False)
)

print('Linhas com chave natural completa:', int(chave_completa.sum()))
print('Linhas em grupos repetidos:', int(duplicada_chave.sum()))
print('Quantidade de grupos repetidos:', len(grupos_duplicados))
display(grupos_duplicados.head(10).to_frame())


Nenhuma linha é removida por duplicidade nesta etapa. Uma decisão desse tipo exigiria confirmar a semântica da chave e examinar proveniência, situação e horários dos registros envolvidos.


## 9. Auditoria temporal e dos atrasos

Calculamos o atraso e a duração planejada apenas para auditoria. As categorias abaixo não são mutuamente exclusivas: um atraso superior a sete dias também é superior a 24 horas. A finalidade é localizar diferentes níveis de severidade, não somá-los como exclusões.

Duração planejada superior a 24 horas é tratada como sinal diagnóstico, não como regra operacional universal.


In [ ]:
atraso_auditado = (
    (df_auditado['dt_partida_real'] - df_auditado['dt_partida_prevista'])
    .dt.total_seconds().div(60)
)
duracao_planejada_auditada = (
    (df_auditado['dt_chegada_prevista'] - df_auditado['dt_partida_prevista'])
    .dt.total_seconds().div(60)
)
atraso_informado = pd.to_numeric(df_auditado['atraso_partida_min'], errors='coerce')
divergencia_atraso = (
    atraso_informado.notna()
    & atraso_auditado.notna()
    & atraso_informado.sub(atraso_auditado).abs().gt(1e-6)
)

flags_temporais = {
    'Antecipação inferior a -24h': atraso_auditado.lt(-1440),
    'Atraso superior a 24h': atraso_auditado.gt(1440),
    'Atraso superior a 7 dias': atraso_auditado.gt(10080),
    'Duração planejada negativa': duracao_planejada_auditada.lt(0),
    'Duração planejada superior a 24h': duracao_planejada_auditada.gt(1440),
    'Atraso informado diverge do recalculado': divergencia_atraso,
}
display(pd.Series({nome: int(flag.sum()) for nome, flag in flags_temporais.items()}, name='Registros').to_frame())


In [ ]:
caso_suspeito = np.logical_or.reduce([flag.fillna(False) for flag in flags_temporais.values()])
campos_rastreio = [
    'arquivo_origem', 'linha_origem', 'sg_empresa_icao', 'nr_voo',
    'sg_icao_origem', 'sg_icao_destino', 'dt_partida_prevista', 'dt_partida_real',
]
amostra_extremos = df_auditado.loc[caso_suspeito, campos_rastreio].copy()
amostra_extremos['atraso_recalculado_min'] = atraso_auditado.loc[caso_suspeito]
amostra_extremos['atraso_informado_min'] = atraso_informado.loc[caso_suspeito]
amostra_extremos['duracao_planejada_min'] = duracao_planejada_auditada.loc[caso_suspeito]

print('Registros sinalizados por ao menos uma auditoria:', int(caso_suspeito.sum()))
display(amostra_extremos.head(20))


Os extremos permanecem na base consolidada. Diferenças muito grandes podem representar problemas de calendário ou consolidação, mas não identificam por si só a causa. Qualquer correção ou exclusão futura deverá usar uma regra verificável e informar quantos registros foram afetados.


## 10. População elegível e reconciliação das exclusões

Recebem alvo apenas voos realizados com as duas datas de partida válidas e partida prevista em 2024 ou 2025. Para evitar dupla contagem, cada registro recebe exatamente um motivo conforme a precedência: situação operacional, disponibilidade das datas e período.


In [ ]:
situacao_normalizada = (
    df_auditado['cd_situacao_voo'].astype('string').str.strip().str.upper()
)
realizado = situacao_normalizada.isin(['REALIZADO', 'REALIZADA'])
partida_valida = (
    df_auditado['dt_partida_prevista'].notna()
    & df_auditado['dt_partida_real'].notna()
)
periodo_valido = df_auditado['dt_partida_prevista'].dt.year.isin([2024, 2025])

motivo_exclusao = pd.Series('Retido', index=df_auditado.index, dtype='string')
motivo_exclusao.loc[~realizado] = 'Situação diferente de realizado ou ausente'
motivo_exclusao.loc[realizado & ~partida_valida] = 'Realizado sem ambas as datas de partida válidas'
motivo_exclusao.loc[realizado & partida_valida & ~periodo_valido] = 'Data prevista fora de 2024–2025'

resumo_populacao = motivo_exclusao.value_counts().rename('Registros').to_frame()
resumo_populacao['Percentual (%)'] = resumo_populacao['Registros'].div(len(df_auditado)).mul(100)
display(resumo_populacao)

if resumo_populacao['Registros'].sum() != len(df_auditado):
    raise AssertionError('A reconciliação da população não fecha com a entrada.')

mascara_elegivel = motivo_exclusao.eq('Retido')
df_elegivel = df_auditado.loc[mascara_elegivel].copy()
print(f'Entrada: {len(df_auditado):,} | elegíveis: {len(df_elegivel):,} | excluídos: {(~mascara_elegivel).sum():,}')


A soma das categorias deve ser exatamente igual ao total de entrada. Cancelamentos e demais situações não realizadas continuam contabilizados, mas não recebem faixa de atraso.


## 11. Construção e teste das variáveis-alvo

As seis faixas são uma definição operacional deste estudo, não categorias regulatórias. A função preserva ausentes como ausentes; nenhum valor inválido pode cair silenciosamente na última faixa.


In [ ]:
FAIXAS = {
    0: 'Pontual ou antecipado',
    1: 'Atraso inferior a 15 min',
    2: 'Atraso de 15 a 30 min',
    3: 'Atraso superior a 30 até 45 min',
    4: 'Atraso superior a 45 até 60 min',
    5: 'Atraso superior a 60 min',
}

def classificar_faixas(valores):
    serie = valores if isinstance(valores, pd.Series) else pd.Series(valores)
    numericos = pd.to_numeric(serie, errors='coerce')
    validos = numericos.notna()
    codigos = np.select(
        [
            validos & numericos.le(0), validos & numericos.lt(15),
            validos & numericos.le(30), validos & numericos.le(45),
            validos & numericos.le(60),
        ],
        [0, 1, 2, 3, 4],
        default=5,
    )
    resultado = pd.Series(codigos, index=numericos.index, dtype='Int64')
    return resultado.mask(~validos, pd.NA)

limites = [
    -1, 0, np.nextafter(0.0, 1.0), np.nextafter(15.0, 0.0), 15,
    30, np.nextafter(30.0, np.inf), 45, np.nextafter(45.0, np.inf),
    60, np.nextafter(60.0, np.inf), np.nan,
]
esperado = [0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5]
resultado_limites = classificar_faixas(limites)
assert resultado_limites.iloc[:-1].tolist() == esperado
assert pd.isna(resultado_limites.iloc[-1])
print('Fronteiras das seis faixas e tratamento de ausentes: validados.')


In [ ]:
df_analitico = df_elegivel.copy()
df_analitico['atraso_partida_minutos'] = (
    (df_analitico['dt_partida_real'] - df_analitico['dt_partida_prevista'])
    .dt.total_seconds().div(60)
)
df_analitico['codigo_faixa_atraso'] = classificar_faixas(
    df_analitico['atraso_partida_minutos']
)
df_analitico['faixa_atraso_partida'] = df_analitico['codigo_faixa_atraso'].map(FAIXAS)
df_analitico['atraso_bi'] = df_analitico['atraso_partida_minutos'].ge(15).astype('Int64')

if df_analitico[['codigo_faixa_atraso', 'faixa_atraso_partida', 'atraso_bi']].isna().any().any():
    raise AssertionError('A população elegível recebeu alvo ausente.')
if not df_analitico['atraso_bi'].eq(df_analitico['codigo_faixa_atraso'].ge(2).astype('Int64')).all():
    raise AssertionError('O alvo binário não corresponde às faixas 2 a 5.')

print('Registros analíticos:', f'{len(df_analitico):,}')


## 12. Variáveis analíticas e risco de vazamento

Calendário, horário previsto e rota são derivados de informações planejadas. Horários reais, situação final, atrasos observados e todas as representações do alvo são pós-evento e não podem entrar em uma futura matriz preditora pré-partida.


In [ ]:
df_analitico['ano'] = df_analitico['dt_partida_prevista'].dt.year
df_analitico['mes'] = df_analitico['dt_partida_prevista'].dt.month
df_analitico['dia_semana'] = df_analitico['dt_partida_prevista'].dt.dayofweek
df_analitico['hora_partida_prevista'] = df_analitico['dt_partida_prevista'].dt.hour
df_analitico['minuto_partida_previsto'] = df_analitico['dt_partida_prevista'].dt.minute
df_analitico['tempo_voo_planejado_minutos'] = (
    (df_analitico['dt_chegada_prevista'] - df_analitico['dt_partida_prevista'])
    .dt.total_seconds().div(60)
)
df_analitico['rota'] = (
    df_analitico['sg_icao_origem'].astype('string')
    + '-'
    + df_analitico['sg_icao_destino'].astype('string')
)

PREDITORES_CANDIDATOS = [
    'sg_empresa_icao', 'nr_voo', 'sg_icao_origem', 'sg_icao_destino',
    'rota', 'mes', 'dia_semana', 'hora_partida_prevista', 'minuto_partida_previsto',
]
print('Hipóteses de preditores pré-partida:', PREDITORES_CANDIDATOS)
print('Excluir de X: datas reais, situação final, atrasos e representações dos alvos.')


## 13. Estatísticas descritivas

Média e desvio-padrão são sensíveis a extremos; mediana, quartis e percentis ajudam a descrever a região central e as caudas. Assimetria e curtose caracterizam a forma observada, mas não explicam suas causas.


In [ ]:
percentis = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
estatisticas_atraso = df_analitico['atraso_partida_minutos'].describe(percentiles=percentis)
tabela_atraso = estatisticas_atraso.rename('Valor').to_frame()
tabela_atraso.loc['IQR', 'Valor'] = estatisticas_atraso['75%'] - estatisticas_atraso['25%']
tabela_atraso.loc['Assimetria', 'Valor'] = df_analitico['atraso_partida_minutos'].skew()
tabela_atraso.loc['Curtose', 'Valor'] = df_analitico['atraso_partida_minutos'].kurtosis()
display(tabela_atraso)

duracoes_validas = df_analitico['tempo_voo_planejado_minutos'].dropna()
duracoes_diagnosticas = duracoes_validas.between(0, 1440)
print('Durações planejadas disponíveis:', f'{len(duracoes_validas):,}')
print('Durações no intervalo diagnóstico de 0 a 24h:', f'{duracoes_diagnosticas.sum():,}')
display(duracoes_validas.loc[duracoes_diagnosticas].describe().rename('Valor').to_frame())


Valores negativos indicam antecipação registrada e valores positivos indicam atraso. Extremos permanecem no cálculo completo e devem ser lidos junto com a auditoria de proveniência. A descrição da duração usa apenas um intervalo diagnóstico explícito e não altera a população do atraso.


## 14. Distribuição das variáveis-alvo

A tabela apresenta contagem, proporção e proporção acumulada nas seis classes ordinais. O gráfico evidencia o desbalanceamento que deverá orientar as métricas da Avaliação 2.


In [ ]:
ordem_codigos = list(FAIXAS)
ordem_rotulos = [FAIXAS[codigo] for codigo in ordem_codigos]
contagem_faixas = (
    df_analitico['codigo_faixa_atraso']
    .value_counts()
    .reindex(ordem_codigos, fill_value=0)
)
proporcao_faixas = contagem_faixas.div(len(df_analitico)).mul(100)
distribuicao_faixas = pd.DataFrame({
    'Código': ordem_codigos,
    'Faixa': ordem_rotulos,
    'Registros': contagem_faixas.values,
    'Proporção (%)': proporcao_faixas.round(2).values,
    'Proporção acumulada (%)': proporcao_faixas.cumsum().round(2).values,
})
display(distribuicao_faixas)

distribuicao_binaria = (
    df_analitico['atraso_bi'].value_counts().reindex([0, 1], fill_value=0)
    .rename('Registros').to_frame()
)
distribuicao_binaria['Proporção (%)'] = distribuicao_binaria['Registros'].div(len(df_analitico)).mul(100)
display(distribuicao_binaria)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
cores = ['#2ca02c', '#66c2a5', '#ffd92f', '#fc8d62', '#e78ac3', '#d62728']
barras = ax.bar(ordem_rotulos, contagem_faixas.values, color=cores)
for barra, percentual in zip(barras, proporcao_faixas.values):
    ax.annotate(
        f'{int(barra.get_height()):,}\n({percentual:.1f}%)',
        (barra.get_x() + barra.get_width() / 2, barra.get_height()),
        ha='center', va='bottom', fontsize=9,
    )
ax.set_title('Distribuição das seis faixas de atraso na partida')
ax.set_xlabel('Faixa ordinal de atraso')
ax.set_ylabel('Quantidade de voos')
ax.tick_params(axis='x', rotation=20)
ax.margins(y=0.15)
plt.tight_layout()
plt.show()


A classe de maior frequência deve ser comparada às classes minoritárias usando as proporções exibidas, não apenas a altura visual das barras. O desequilíbrio não determina antecipadamente qual modelo será melhor, mas exige métricas por classe, macro-F1, acurácia balanceada e matriz de confusão na Avaliação 2.


## 15. Forma da distribuição contínua

Para preservar legibilidade, a visualização usa o intervalo de −30 a 180 minutos. Esse é somente um recorte gráfico: nenhuma observação é removida de `df_analitico`, e a quantidade fora do intervalo é informada.


In [ ]:
atraso_visual = df_analitico.loc[
    df_analitico['atraso_partida_minutos'].between(-30, 180),
    'atraso_partida_minutos',
]
print('No recorte visual:', f'{len(atraso_visual):,}')
print('Fora do recorte visual:', f'{len(df_analitico) - len(atraso_visual):,}')

if atraso_visual.empty:
    raise ValueError('Não há observações no intervalo visual definido.')

fig, axes = plt.subplots(2, 1, figsize=(13, 8), gridspec_kw={'height_ratios': [1, 2]})
sns.boxplot(x=atraso_visual, showfliers=False, color='#72B6A1', ax=axes[0])
axes[0].set(title='Região central do atraso registrado', xlabel='Atraso na partida (minutos)')
sns.histplot(atraso_visual, bins=60, color='#3576A3', ax=axes[1])
for limite, cor, rotulo in [
    (0, '#338C62', '0 min'), (15, '#D98B18', '15 min'), (60, '#C34B4B', '60 min'),
]:
    axes[1].axvline(limite, color=cor, linestyle='--', linewidth=2, label=rotulo)
axes[1].set(title='Distribuição no recorte visual', xlabel='Atraso na partida (minutos)', ylabel='Voos')
axes[1].legend(title='Limites')
plt.tight_layout()
plt.show()


O boxplot omite marcadores individuais de extremos somente para facilitar a leitura da caixa. A auditoria anterior continua sendo a fonte para investigar as caudas e os registros rastreáveis.


## 16. Evolução mensal e padrão por hora prevista

As agregações seguintes mostram volume, mediana e proporção de atrasos de pelo menos 15 minutos. Elas descrevem variações temporais na população observada; não isolam sazonalidade, composição de rotas ou outros fatores.


In [ ]:
resumo_mensal = (
    df_analitico
    .groupby(['ano', 'mes'])
    .agg(
        Registros=('atraso_bi', 'size'),
        Taxa_atraso=('atraso_bi', 'mean'),
        Mediana_min=('atraso_partida_minutos', 'median'),
    )
    .reset_index()
)
resumo_mensal['Período'] = pd.to_datetime(
    dict(year=resumo_mensal['ano'], month=resumo_mensal['mes'], day=1)
)
resumo_mensal['Taxa de atraso (%)'] = resumo_mensal['Taxa_atraso'].mul(100)
display(resumo_mensal[['Período', 'Registros', 'Taxa de atraso (%)', 'Mediana_min']])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(resumo_mensal['Período'], resumo_mensal['Taxa de atraso (%)'], marker='o', color='#8C2D4A')
ax.set(title='Atrasos ≥15 min por mês previsto', xlabel='Mês da partida prevista', ylabel='Taxa (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
resumo_hora = (
    df_analitico
    .groupby('hora_partida_prevista')['atraso_bi']
    .agg(Registros='size', Atrasados='sum', Taxa='mean')
)
resumo_hora['Taxa (%)'] = resumo_hora['Taxa'].mul(100)
display(resumo_hora[['Registros', 'Atrasados', 'Taxa (%)']])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(resumo_hora.index, resumo_hora['Taxa (%)'], marker='o', color='#B2182B')
ax.set(title='Atrasos ≥15 min por hora prevista', xlabel='Hora prevista', ylabel='Taxa (%)')
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()


As tabelas fornecem os denominadores de cada ponto. Diferenças entre meses ou horas são associações descritivas e podem refletir mudanças na composição de companhias, aeroportos e rotas.


## 17. Comparações por companhia, aeroportos e rota

Para reduzir rankings instáveis, selecionamos até dez grupos entre os de maior volume e exigimos pelo menos 1.000 registros. O limite é analítico e deve ser reavaliado na modelagem; aqui ele garante que taxa e denominador sejam apresentados juntos.


In [ ]:
def resumir_grupo(coluna, volume_minimo=1000, quantidade=10):
    resumo = (
        df_analitico
        .dropna(subset=[coluna])
        .groupby(coluna)['atraso_bi']
        .agg(Registros='size', Atrasados='sum', Taxa='mean')
    )
    resumo = resumo.loc[resumo['Registros'].ge(volume_minimo)]
    resumo = resumo.nlargest(quantidade, 'Registros').copy()
    resumo['Taxa (%)'] = resumo['Taxa'].mul(100)
    return resumo[['Registros', 'Atrasados', 'Taxa (%)']]

resumos_grupos = {
    'Companhia': resumir_grupo('sg_empresa_icao'),
    'Origem': resumir_grupo('sg_icao_origem'),
    'Destino': resumir_grupo('sg_icao_destino'),
    'Rota': resumir_grupo('rota'),
}
for titulo, resumo in resumos_grupos.items():
    print(titulo, '— grupos elegíveis:', len(resumo))
    display(resumo)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, (titulo, resumo) in zip(axes.flat, resumos_grupos.items()):
    if resumo.empty:
        ax.text(0.5, 0.5, 'Sem grupos com volume mínimo', ha='center', va='center')
        ax.set_axis_off()
        continue
    resumo.sort_values('Taxa (%)')['Taxa (%)'].plot(kind='barh', color='#2B8CBE', ax=ax)
    ax.set_title(f'Atrasos ≥15 min — {titulo.lower()}')
    ax.set_xlabel('Taxa (%)')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()


Os gráficos comparam apenas grupos de grande volume e as tabelas preservam seus denominadores. Uma taxa diferente não prova que companhia, aeroporto ou rota cause atraso; composição temporal, malha e características operacionais podem estar associadas simultaneamente.


## 18. Discussão preliminar

**População e exclusões.** O alvo descreve voos realizados de 2024 e 2025 com horários de partida interpretáveis. Cancelamentos, datas inválidas e registros fora do período são contabilizados em categorias exclusivas, permitindo reconciliar entrada e população final.

**Qualidade.** A análise quantifica ausências, domínios, duplicidades, falhas de conversão, extremos e divergências entre atraso informado e recalculado. Os registros sinalizados permanecem rastreáveis; não foi aplicada correção automática de calendário nem exclusão arbitrária das caudas.

**Distribuição.** As estatísticas e as seis faixas revelam concentração desigual entre as classes. A Avaliação 2 deverá usar métricas por classe e comparar modelos com uma referência simples, evitando depender apenas de acurácia global.

**Padrões exploratórios.** Mês, hora prevista, companhia, aeroportos e rotas apresentam associações descritivas. Os denominadores e o volume mínimo reduzem interpretações baseadas em grupos muito pequenos, mas não controlam fatores de confusão nem estabelecem causalidade.

**Vazamento e alta cardinalidade.** Informações reais e representações do atraso são pós-evento. Identificadores de companhia, aeroporto, rota e voo exigirão codificação e validação temporal cuidadosas; transformações que aprendam parâmetros deverão ser ajustadas somente no treino.

**Limites.** A base consolidada não equivale aos arquivos mensais brutos da ANAC. A validade das conclusões depende da cobertura, dos critérios de consolidação e da consistência dos timestamps. Os extremos devem ser investigados antes de qualquer tratamento definitivo.


## 19. Conclusão e encaminhamento

A Avaliação 1 estabelece uma população analítica reproduzível, documenta as seis faixas de atraso na partida e evidencia os principais riscos de qualidade e desbalanceamento. As análises descrevem os registros disponíveis e não medem capacidade preditiva nem causalidade.

Na Avaliação 2, treino, validação e teste deverão respeitar a ordem temporal. A matriz de preditores poderá usar somente informações disponíveis antes da partida; horários reais, situação final, atrasos e alvos serão excluídos. A seleção de algoritmos, codificações e estratégias para classes desbalanceadas dependerá do desempenho observado na validação.
